In [1]:
import json
import numpy as np
import pandas as pd
from pandas import json_normalize
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from transformers import CamembertTokenizer, CamembertModel
import torch
from tqdm import tqdm

/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Data loading and exploration

In [2]:
TEXT_COL = "full_text"
DESC_COL = "user.description"
ID_COL = "challenge_id"
DATA_DIR = "./data"
EMB_DIR = os.path.join(DATA_DIR, "embeddings")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EMB_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 16

# 1. Load JSONL files
# ============================

train = pd.read_json("data/train.jsonl", lines=True)
train = json_normalize(train.to_dict(orient="records"))

X_kaggle = pd.read_json("data/kaggle_test.jsonl", lines=True)
X_kaggle = json_normalize(X_kaggle.to_dict(orient="records"))

X_train = train.drop("label", axis=1)
y_train = train["label"]
np.save(os.path.join(DATA_DIR, "y_train.npy"), y_train)


In [3]:
# 2. Extract text
# ============================

def extract_full_text(row):
    txt = row["text"]
    if not pd.isna(row.get("extended_tweet.full_text", np.nan)):
        txt = row["extended_tweet.full_text"]
    return txt

X_train[TEXT_COL] = X_train.apply(extract_full_text, axis=1)
X_kaggle[TEXT_COL] = X_kaggle.apply(extract_full_text, axis=1)

# 2. Data pre-processing

In [4]:
# 3. Drop list/dict columns
# ============================

def drop_list_columns(df):
    bad = []
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, (list, dict))).any():
            bad.append(col)
    return df.drop(columns=bad)

X_train_clean = drop_list_columns(X_train)
X_kaggle_clean = drop_list_columns(X_kaggle)

# ============================
# 4. Synchronize columns
# ============================

common_cols = list(set(X_train_clean.columns) & set(X_kaggle_clean.columns))

if ID_COL not in common_cols:
    print("WARNING: ID column missing in one dataset!")

# Keep text column separate
if TEXT_COL in common_cols:
    common_cols.remove(TEXT_COL)

# Final column lists
X_train_clean = X_train_clean[[TEXT_COL] + common_cols]
X_kaggle_clean = X_kaggle_clean[[TEXT_COL] + common_cols]

In [5]:
# 5. Feature Engineering
# ============================
remaining_cols = [
    col for col in X_train_clean.columns
]

print(remaining_cols)

for col in remaining_cols:
    print(col, X_train_clean[col].nunique())
    print(X_train_clean[col].unique()[:10])
    print()

# Numerical features that have Nan as values (--> 0)
to_numeric_cols = ["quoted_status.user.favourites_count", "quoted_status.favorite_count", "quoted_status.reply_count", "quoted_status.user.friends_count", "quoted_status.quote_count", "quoted_status.user.listed_count", "quoted_status.retweet_count"]
### to_embed = ["extended_tweet.full_text", "user.description", "full_text"] A VOIR PLUS TARD QUOI EN FAIRE 

# Categorical features to OneHot Encode
categorical_cols = ["quoted_status.user.geo_enabled", "quoted_status.user.is_translator", "quoted_status.is_quote_status", "quoted_status.scopes.followers", "user.profile_use_background_image", "is_quote_status", "user.geo_enabled", "truncated", "place.place_type", "possibly_sensitive", "user.is_translator", "quoted_status.user.profile_use_background_image", "quoted_status.user.default_profile_image", "quoted_status.retweeted", "geo.type", "quoted_status.geo.type", "quoted_status.user.contributors_enabled", "quoted_status.user.protected", "quoted_status.user.translator_type", "user.translator_type", "user.profile_background_tile", "quoted_status.coordinates.type", "quoted_status.place.place_type", "quoted_status.place.bounding_box.type", "user.default_profile", "coordinates.type" ,"quoted_status.filter_level", "coordinates.type", "user.default_profile",  "quoted_status.place.bounding_box.type", "quoted_status.place.place_type", "quoted_status.coordinates.type", "user.profile_background_tile", "user.translator_type"]
print(f"chosen categorical cols {categorical_cols}") 

# 6. Structured columns
# ============================

structured_cols = [c for c in X_train_clean.columns if c not in [TEXT_COL, ID_COL]]
X_train_clean[to_numeric_cols] = X_train_clean[to_numeric_cols].fillna(0)
X_kaggle_clean[to_numeric_cols] = X_kaggle_clean[to_numeric_cols].fillna(0)
numeric_cols = X_train_clean[structured_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()


['full_text', 'timestamp_ms', 'user.notifications', 'challenge_id', 'created_at', 'quoted_status.reply_count', 'quoted_status.text', 'quoted_status_permalink.url', 'place.place_type', 'quoted_status.extended_tweet.full_text', 'geo', 'reply_count', 'quoted_status.user.profile_sidebar_fill_color', 'quoted_status.user.profile_background_color', 'retweet_count', 'user.created_at', 'quoted_status.lang', 'quoted_status.created_at', 'quoted_status.id', 'user.favourites_count', 'quoted_status.user.name', 'quoted_status.id_str', 'user.translator_type', 'quoted_status.in_reply_to_user_id_str', 'user.lang', 'user.utc_offset', 'quoted_status', 'quoted_status.quoted_status_id_str', 'place', 'favorited', 'quoted_status.in_reply_to_user_id', 'extended_tweet', 'user.time_zone', 'quoted_status.user.favourites_count', 'quote_count', 'quoted_status.retweet_count', 'quoted_status.user.contributors_enabled', 'quoted_status.user.utc_offset', 'quoted_status_permalink.display', 'is_quote_status', 'user.is_tra

In [6]:
print(numeric_cols)

['quoted_status.reply_count', 'geo', 'reply_count', 'retweet_count', 'quoted_status.id', 'user.favourites_count', 'quoted_status', 'place', 'quoted_status.in_reply_to_user_id', 'extended_tweet', 'quoted_status.user.favourites_count', 'quote_count', 'quoted_status.retweet_count', 'quoted_status.user.utc_offset', 'in_reply_to_status_id_str', 'id_str', 'quoted_status.favorite_count', 'quoted_status.user.id', 'quoted_status.geo', 'quoted_status.user.friends_count', 'possibly_sensitive', 'quoted_status.user.follow_request_sent', 'quoted_status.user.notifications', 'contributors', 'quoted_status.user.statuses_count', 'favorite_count', 'quoted_status_permalink', 'quoted_status.user.lang', 'in_reply_to_user_id', 'quoted_status.quote_count', 'quoted_status_id_str', 'quoted_status.user.time_zone', 'user.listed_count', 'coordinates', 'quoted_status.user.listed_count', 'quoted_status.contributors', 'quoted_status.in_reply_to_status_id', 'quoted_status.quoted_status_id', 'quoted_status.place', 'in_

In [7]:
# 7. Preprocess structured features
# ============================

preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="drop"
)

X_train_struct = preprocessor.fit_transform(X_train_clean)
X_kaggle_struct = preprocessor.transform(X_kaggle_clean)

# ============================
# 8. Extract ID arrays
# ============================

train_ids = X_train_clean[ID_COL].astype(float).values.reshape(-1, 1)
kaggle_ids = X_kaggle_clean[ID_COL].astype(float).values.reshape(-1, 1)


/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [8]:
embedding_dir = os.path.join(DATA_DIR, "embeddings")

In [9]:
### DO NOT EXECUTE THIS CELL UNLESS YOU WANT TO UPDATE(OR GENERATE) EMBEDDINGS

# 9. EMBED TEXT USING CAMEMBERT
# ============================
os.makedirs(embedding_dir, exist_ok=True)
train_embedding_path = os.path.join(embedding_dir, "X_train_text_embeddings.npy")
kaggle_embedding_path = os.path.join(embedding_dir, "X_kaggle_text_embeddings.npy")
# -----------------------------------------------------------------

tokenizer = CamembertTokenizer.from_pretrained('camembert-base')
model = CamembertModel.from_pretrained('camembert-base')
model.to(device)
model.eval()

def embed_texts(texts, batch_size=16):
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]
            encoded = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            # Use [CLS] token representation (first token) as embedding
            batch_embeddings = outputs.last_hidden_state[:,0,:].cpu().numpy()
            embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

def embed_texts_multilayer(texts, batch_size=16, layers=[6, 9, 12]):
    """Extract embeddings from multiple transformer layers for richer representations."""
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]
            encoded = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            # Get hidden states from all layers
            outputs = model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
            
            # Extract [CLS] token from specified layers and concatenate
            layer_embeddings = []
            for layer_idx in layers:
                layer_emb = outputs.hidden_states[layer_idx][:, 0, :].cpu().numpy()
                layer_embeddings.append(layer_emb)
            
            # Concatenate horizontally: 3 layers × 768 = 2304 dimensions
            batch_embeddings = np.hstack(layer_embeddings)
            embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

# Combine main text + description into one field
train_texts = (
    X_train_clean[TEXT_COL].astype(str) 
    +"\nUser description: "
    + X_train_clean[DESC_COL].astype(str)
).tolist()

kaggle_texts = (
    X_kaggle_clean[TEXT_COL].astype(str) 
    + "\nUser description: "
    + X_kaggle_clean[DESC_COL].astype(str)
).tolist()


# Compute the embeddings
print("Embedding TRAIN texts...")
train_embeddings = embed_texts(train_texts, batch_size=batch_size)
np.save(train_embedding_path, train_embeddings)
print("Train embeddings saved to:", train_embedding_path)

print("Embedding KAGGLE texts...")
kaggle_embeddings = embed_texts(kaggle_texts, batch_size=batch_size)
np.save(kaggle_embedding_path, kaggle_embeddings)
print("Kaggle embeddings saved to:", kaggle_embedding_path)



Embedding TRAIN texts...


100%|██████████| 9683/9683 [05:44<00:00, 28.08it/s]


Train embeddings saved to: ./data/embeddings/X_train_text_embeddings.npy
Embedding KAGGLE texts...


100%|██████████| 6462/6462 [03:51<00:00, 27.97it/s]


Kaggle embeddings saved to: ./data/embeddings/X_kaggle_text_embeddings.npy


In [ ]:
### DO NOT EXECUTE THIS CELL UNLESS YOU WANT TO UPDATE(OR GENERATE) MULTILAYER EMBEDDINGS

# ============================
# 10. MULTI-LAYER EMBEDDINGS
# ============================
print("\n" + "="*70)
print("EXTRACTING MULTI-LAYER EMBEDDINGS")
print("="*70)
print("Extracting from layers 6, 9, and 12 for richer representations...")

train_multilayer_path = os.path.join(embedding_dir, "X_train_multilayer_embeddings.npy")
kaggle_multilayer_path = os.path.join(embedding_dir, "X_kaggle_multilayer_embeddings.npy")

print("\nEmbedding TRAIN texts (multi-layer)...")
train_multilayer_embeddings = embed_texts_multilayer(train_texts, batch_size=batch_size, layers=[6, 9, 12])
np.save(train_multilayer_path, train_multilayer_embeddings)
print(f"Train multi-layer embeddings saved: {train_multilayer_embeddings.shape}")
print(f"Saved to: {train_multilayer_path}")

print("\nEmbedding KAGGLE texts (multi-layer)...")
kaggle_multilayer_embeddings = embed_texts_multilayer(kaggle_texts, batch_size=batch_size, layers=[6, 9, 12])
np.save(kaggle_multilayer_path, kaggle_multilayer_embeddings)
print(f"Kaggle multi-layer embeddings saved: {kaggle_multilayer_embeddings.shape}")
print(f"Saved to: {kaggle_multilayer_path}")

print("\n" + "="*70)
print("Multi-layer embeddings extraction complete!")
print(f"Single-layer: 768 dimensions")
print(f"Multi-layer: {train_multilayer_embeddings.shape[1]} dimensions (3 × 768)")
print("="*70)


In [ ]:
train_embedding_path = os.path.join(embedding_dir, "X_train_text_embeddings.npy")
kaggle_embedding_path = os.path.join(embedding_dir, "X_kaggle_text_embeddings.npy")

X_train_text = np.load(train_embedding_path)
X_kaggle_text = np.load(kaggle_embedding_path)

print("Loaded text embeddings:")
print("train:", X_train_text.shape)
print("kaggle:", X_kaggle_text.shape)


train_full_path = os.path.join(DATA_DIR, "X_train_processed.npy")
kaggle_full_path = os.path.join(DATA_DIR, "X_kaggle_processed.npy")


print("Concatenating structured features and text embeddings...")

# Horizontal concatenation
X_train_full = np.hstack([X_train_struct, X_train_text])
X_kaggle_full = np.hstack([X_kaggle_struct, X_kaggle_text])

# Save processed arrays
size_bytes = X_train_full.nbytes
print(size_bytes / (1024**3), "GB")
np.save(train_full_path, X_train_full)
np.save(kaggle_full_path, X_kaggle_full)
print("Final feature arrays saved.")
print("  train:", X_train_full.shape)
print("  kaggle:", X_kaggle_full.shape)

# ============================
# 11. MULTI-LAYER CONCATENATION
# ============================
print("\n" + "="*70)
print("CREATING MULTI-LAYER FEATURE ARRAYS")
print("="*70)

train_multilayer_path = os.path.join(embedding_dir, "X_train_multilayer_embeddings.npy")
kaggle_multilayer_path = os.path.join(embedding_dir, "X_kaggle_multilayer_embeddings.npy")

X_train_multilayer = np.load(train_multilayer_path)
X_kaggle_multilayer = np.load(kaggle_multilayer_path)

print("Loaded multi-layer embeddings:")
print("  train:", X_train_multilayer.shape)
print("  kaggle:", X_kaggle_multilayer.shape)

train_multilayer_full_path = os.path.join(DATA_DIR, "X_train_processed_multilayer.npy")
kaggle_multilayer_full_path = os.path.join(DATA_DIR, "X_kaggle_processed_multilayer.npy")

print("\nConcatenating structured features + multi-layer embeddings (2304)...")

# Horizontal concatenation: 8 + 2304 = 2312 total features
X_train_multilayer_full = np.hstack([X_train_struct, X_train_multilayer])
X_kaggle_multilayer_full = np.hstack([X_kaggle_struct, X_kaggle_multilayer])


# Save processed arrays
np.save(train_multilayer_full_path, X_train_multilayer_full)
np.save(kaggle_multilayer_full_path, X_kaggle_multilayer_full)

print(f"\nMulti-layer feature arrays saved:")
print(f"  Train shape: {X_train_multilayer_full.shape}")
print(f"  Kaggle shape: {X_kaggle_multilayer_full.shape}")
print(f"  Saved to: {train_multilayer_full_path}")
print(f"  Saved to: {kaggle_multilayer_full_path}")
print("="*70)


Loaded text embeddings:
train: (154914, 768)
kaggle: (103380, 768)
Concatenating structured features and text embeddings...
1.0387793183326721 GB
Final feature arrays saved.
  train: (154914, 900)
  kaggle: (103380, 900)

CREATING MULTI-LAYER FEATURE ARRAYS
Loaded multi-layer embeddings:
  train: (154914, 2304)
  kaggle: (103380, 2304)

Concatenating structured features + multi-layer embeddings (2304)...

Multi-layer feature arrays saved:
  Train shape: (154914, 2436)
  Kaggle shape: (103380, 2436)
  Saved to: ./data/X_train_processed_multilayer.npy
  Saved to: ./data/X_kaggle_processed_multilayer.npy


: 